<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 15


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс OrderLine в C#, который будет представлять информацию о
строке заказа, содержащей детали одного товара в заказе. На основе этого класса
разработать 2-3 производных класса, демонстрирующих принципы наследования и
полиморфизма. В каждом из классов должны быть реализованы новые атрибуты и
методы, а также переопределены некоторые методы базового класса для
демонстрации полиморфизма.

Требования к базовому классу OrderLine:
• Атрибуты: ID товара (ProductId), Название товара (ProductName), Цена
товара (Price).
• Методы:
o
o CalculateTotal(): метод для расчета общей стоимости строки заказа.
o UpdatePrice(decimal newPrice): метод для обновления цены товара в
строке заказа.
o GetProductDetails(): метод для получения деталей товара.

Требования к производным классам:
1. СтандартнаяСтрока (StandardLine): Должна содержать дополнительные
атрибуты, такие как Количество единиц (Units). Метод CalculateTotal() должен
быть переопределен для учета количества единиц при расчете общей
стоимости.
2. СпециальнаяСтрока (SpecialLine): Должна содержать дополнительные
атрибуты, такие как Скидка (Discount). Метод UpdatePrice() должен быть
переопределен для применения скидки к цене товара.
3. БесплатнаяСтрока (FreeLine) (если требуется третий класс): Должна
содержать дополнительные атрибуты, такие как Предварительный платеж
(Prepayment). Метод CalculateTotal() должен быть переопределен для учета
предварительного плата при расчете общей стоимости.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [4]:
using System;
public interface IDate
{
    public void Dat();
}
// Делегат для событий изменения цены
public delegate void PriceChangedEventHandler(object sender, PriceChangedEventArgs e);
// Аргументы события изменения цены
public class PriceChangedEventArgs : EventArgs
{
    public decimal OldPrice { get; set; }
    public decimal NewPrice { get; set; }
    public string ProductId { get; set; }
    
    public PriceChangedEventArgs(string productId, decimal oldPrice, decimal newPrice)
    {
        ProductId = productId;
        OldPrice = oldPrice;
        NewPrice = newPrice;
    }
}
public class OrderLine: IDate
{    
    public event PriceChangedEventHandler PriceChanged;
    public string ProductId { get; set; }
    public string ProductName { get; set; }
    public decimal Price { get; set; }
    // дополнительные атрибуты
    public string Category{get;set;}
    public string Supplier{get;set;}
    public DateTime Date{get;set;}
    
    public OrderLine(string productid,string productname,decimal price,string category,string supplier)
    {
        ProductId=productid;
        ProductName=productname;
        Price=price;
        Category = category;
        Supplier = supplier;
    }
    public virtual void CalculateTotal()
    {
        Console.WriteLine($"Общая стоимость заказа: {Price}");
    }
    public virtual void UpdatePrice(decimal newPrice)
    {
        if (newPrice != Price)
        {
            var oldPrice = Price;
            Price = newPrice;
            OnPriceChanged(new PriceChangedEventArgs(ProductId, oldPrice, newPrice));
        }
    }
    protected virtual void OnPriceChanged(PriceChangedEventArgs e)
    {
        PriceChanged?.Invoke(this, e);
    }
    public void GetProductDetails()
    {
        Console.WriteLine( $"Id товара: {ProductId}, Название товара: {ProductName}, Цена: {Price}");
    }
    //дополнительные методы
    public virtual void SupplierInfo()
    {
        Console.WriteLine($"Поставщик: {Supplier}");
    }
    public virtual void Dat()
    {
        Console.WriteLine($"Дата создания заказа: {DateTime.Now}");
    }
    public virtual void Catt()
    {
        Console.WriteLine($"Категория товара: {Category}");
    }
}
public class StandardLine: OrderLine, IDate
{
    public int Units{get;set;}
    public StandardLine(string productid,string productname,decimal price,string category,string supplier,int units) : base(productid,productname,price,supplier,category)
    {
        Units = units;
    }
    public override void CalculateTotal()
    {
        Console.WriteLine($"Количество единиц товара:{Units}. Стоимость одной единицы товара: {Math.Round(Price/Units)}");
    }
    // новые методы
    public void UpdatePrice(int newUnits)
    {
        Units = newUnits;
        Console.WriteLine($"Новое количество единиц товара: {newUnits}");
    }
    public override void Dat()
    {
        Console.WriteLine($"Дата обновления количества товара: {DateTime.Now}");
    }

}
public class SpecialLine: OrderLine
{
    public int Discount{get;set;}
    public SpecialLine(string productid,string productname,decimal price,string category,string supplier,int discount): base(productid,productname,price,supplier,category)
    {
        Discount = discount;
    }
    public void UpdatePrice()
    {
        Console.WriteLine($"Скидка:{Discount}%."); 
    }
}
public class FreeLine: OrderLine
{
    public int Prepayment{get;set;}
    public FreeLine(string productid,string productname,decimal price,string category,string supplier,int prepayment): base(productid,productname,price,supplier,category)
    {
        Prepayment = prepayment;
    }
    public override void CalculateTotal()
    {
        Console.WriteLine($"Предварительный платеж:{Prepayment} Оставшаяся сумма: {Price-Prepayment}");
    }

}
//Измененная колекция заказов с внедрением зависимости
public class OrderCollection<T> where T : OrderLine, IDate
{
    private List<T> _orders = new List<T>();
    // Событие для уведомления об изменениях в заказе
    private event Action<string> _notify;
    public OrderCollection(Action<string> notify = null)
    {
        _notify = notify ?? Console.WriteLine; // Значение по умолчанию
    }
    public void AddOrder(T order)
    {
        _orders.Add(order);
        _notify?.Invoke($"Заказ добавлен: {order.ProductName}");
    }
    
    public void RemoveOrder(T order)
    {
        if (_orders.Remove(order))
        {
            _notify?.Invoke($"Заказ удален: {order.ProductName}");
        }
    }
    
    public void DisplayAllOrders()
    {
        Console.WriteLine($"\nВсего заказов в списке: {_orders.Count}");
        foreach (var order in _orders)
        {
            order.GetProductDetails();
        }
    }
    public void CalculateTotalPrice()
    {
        decimal total = 0;
        foreach (var order in _orders)
        {
            total += order.Price;
        }
        Console.WriteLine($"Стоимость всех товаров в списке: {total}");
    }
}

OrderCollection<OrderLine> orderCollection = new OrderCollection<OrderLine>();

OrderLine product = new OrderLine("i12332","Ноутбук",10000,"Техника","Acer");
product.CalculateTotal();
product.UpdatePrice(15000);
product.GetProductDetails();
product.SupplierInfo();
((IDate)product).Dat();
product.Catt();

StandardLine Units = new StandardLine("i12332","Ноутбук",15000,"Техника","Acer",10);
Units.CalculateTotal();
Units.UpdatePrice(15);
((IDate)Units).Dat();

SpecialLine Discount = new SpecialLine("i12332","Ноутбук",15000,"Техника","Acer",10);;
Discount.UpdatePrice();

NewLine n = new NewLine("i12332","Ноутбук",15000,"Техника","Acer",10,"Чёрная пятница");
n.CalculateTotal();

FreeLine Prepayment = new FreeLine("i12332","Ноутбук",15000,"Техника","Acer",5000);
Prepayment.CalculateTotal();

OrderLine mouse = new OrderLine("i12333", "Мышь", 15000, "Техника", "Logitech");
OrderLine con = new OrderLine("i12334", "Клавиатура", 8000, "Техника", "Razer");
OrderLine mon = new OrderLine("i12335", "Монитор", 20000, "Техника", "Samsung");
OrderLine covric = new OrderLine("i12336", "Коврик", 3000, "Техника", "SteelSeries");

orderCollection.AddOrder(product);
orderCollection.AddOrder(mouse);
orderCollection.AddOrder(con);
orderCollection.AddOrder(mon);
orderCollection.AddOrder(covric);
orderCollection.DisplayAllOrders();
orderCollection.CalculateTotalPrice();

Общая стоимость заказа: 10000
Id товара: i12332, Название товара: Ноутбук, Цена: 15000
Поставщик: Acer
Дата создания заказа: 15.11.2025 12:38:16
Категория товара: Техника
Количество единиц товара:10. Стоимость одной единицы товара: 1500
Новое количество единиц товара: 15
Дата обновления количества товара: 15.11.2025 12:38:16
Скидка:10%.
Акция: Чёрная пятница, Скидка: 10%, Цена товара со скидкой: 13500
Предварительный платеж:5000 Оставшаяся сумма: 10000
Заказ добавлен: Ноутбук
Заказ добавлен: Мышь
Заказ добавлен: Клавиатура
Заказ добавлен: Монитор
Заказ добавлен: Коврик

Всего заказов в списке: 5
Id товара: i12332, Название товара: Ноутбук, Цена: 15000
Id товара: i12333, Название товара: Мышь, Цена: 15000
Id товара: i12334, Название товара: Клавиатура, Цена: 8000
Id товара: i12335, Название товара: Монитор, Цена: 20000
Id товара: i12336, Название товара: Коврик, Цена: 3000
Стоимость всех товаров в списке: 61000
